In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

In [ ]:
# Load Kenya data
df = pd.read_csv('data/kenya.csv')
df['Country'] = 'Kenya'

print(f"Data loaded! Shape: {df.shape}")
df.head()

In [ ]:
# Replace -999 with NaN
df_clean = df.replace(-999, np.nan)

# Check missing values
print(f"Missing values per column:")
print(df_clean.isna().sum())

In [ ]:
# Convert dates
df_clean['Date'] = pd.to_datetime(
    df_clean['YEAR'].astype(str) + df_clean['DOY'].astype(str).str.zfill(3),
    format='%Y%j'
)
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Year'] = df_clean['Date'].dt.year

print(f"Date range: {df_clean['Date'].min()} to {df_clean['Date'].max()}")

In [ ]:
# Forward fill missing values
df_clean = df_clean.fillna(method='ffill').dropna()

# Save cleaned data
df_clean.to_csv('data/kenya_clean.csv', index=False)
print(f"Cleaned data saved! Shape: {df_clean.shape}")

In [ ]:
# Temperature time series
monthly_temp = df_clean.groupby(['Year', 'Month'])['T2M'].mean().reset_index()
monthly_temp['Date'] = pd.to_datetime(monthly_temp['Year'].astype(str) + '-' + monthly_temp['Month'].astype(str))

plt.figure(figsize=(14, 6))
plt.plot(monthly_temp['Date'], monthly_temp['T2M'], 'g-', linewidth=1.5)
plt.title('Kenya: Monthly Average Temperature (2015-2026)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Mean temperature: {df_clean['T2M'].mean():.1f}°C")

In [ ]:
# Precipitation analysis
monthly_precip = df_clean.groupby(['Year', 'Month'])['PRECTOTCORR'].sum().reset_index()
monthly_precip['Date'] = pd.to_datetime(monthly_precip['Year'].astype(str) + '-' + monthly_precip['Month'].astype(str))

plt.figure(figsize=(14, 6))
plt.bar(monthly_precip['Date'], monthly_precip['PRECTOTCORR'], width=25, color='steelblue')
plt.title('Kenya: Monthly Total Precipitation (2015-2026)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Precipitation (mm)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Mean daily rainfall: {df_clean['PRECTOTCORR'].mean():.2f} mm")

In [ ]:
# Summary statistics
print("="*50)
print("KENYA CLIMATE SUMMARY")
print("="*50)
print(f"Average Temperature: {df_clean['T2M'].mean():.1f}°C")
print(f"Max Temperature: {df_clean['T2M_MAX'].max():.1f}°C")
print(f"Average Rainfall: {df_clean['PRECTOTCORR'].mean():.2f} mm/day")
print(f"Dry days (%): {((df_clean['PRECTOTCORR'] < 1).sum() / len(df_clean)) * 100:.1f}%")